In [0]:
#Imports
from pyspark.sql.functions import current_timestamp

In [0]:
#table_name = "credit_bureau_reports"
table_name = "payment_gateway_logs"

In [0]:
df = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format","csv")
            .option("cloudFiles.schemaLocation",f"/Volumes/neo_bank/landing/files/_schema/{table_name}")
            .option("header","True")
            .load(f"/Volumes/neo_bank/landing/files/{table_name}/")
    )

df=df.withColumn("insert_timestamp",current_timestamp())

(
    df.writeStream
        .format("delta")
        .option("checkpointLocation",f"/Volumes/neo_bank/landing/files/_checkpoints/{table_name}")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(f"neo_bank.bronze.{table_name}")
)